In [0]:
dbutils.widgets.text("catalog", "olist_dev")
CATALOG = dbutils.widgets.get("catalog")
LAKEHOUSE = f"{CATALOG}.lakehouse"
TABLE_FEATURES = f"{CATALOG}.ml.features_commandes"
print(f"source : {LAKEHOUSE}\ncible  : {TABLE_FEATURES}")

In [0]:
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from pyspark.sql import functions as F
from olist.bronze import ecrire_table
from olist.features import construire_features, COLONNES_INTERDITES

In [0]:
gold = lambda nom: spark.table(f"{LAKEHOUSE}.{nom}")

features = construire_features(
    gold("fct_orders"), gold("fct_order_items"), gold("dim_customer"),
    gold("dim_seller"), gold("dim_product"), gold("dim_date"),
)
ecrire_table(features, TABLE_FEATURES)
print(f"écrite : {TABLE_FEATURES}")

In [0]:
feat = spark.table(TABLE_FEATURES)
erreurs = []

n = feat.count()
if n != 99_441:
    erreurs.append(f"{n:,} lignes au lieu de 99 441")
if feat.select("order_id").distinct().count() != n:
    erreurs.append("order_id non unique")

fuites = [c for c in COLONNES_INTERDITES if c in feat.columns]
if fuites:
    erreurs.append(f"colonnes interdites présentes : {fuites}")

print(f"lignes : {n:,}   fuites : {fuites or 'aucune'}")
display(
    feat.filter("dans_population")
    .groupBy("periode")
    .agg(F.count("*").alias("commandes"), F.sum("en_retard").alias("retards"),
         F.round(F.avg("en_retard") * 100, 2).alias("taux_retard_pct"))
    .orderBy("periode")
)
assert not erreurs, erreurs

In [0]:
# On prend une commande dont le vendeur a un long historique…
ligne = (feat.filter("dans_population AND vendeur_nb > 50")
             .select("order_id", "vendeur_principal", "order_approved_at",
                     "vendeur_nb", "vendeur_taux_retard")
             .orderBy("order_id").first())

# … et on recompte son historique sans passer par la fonction
fo, fi = gold("fct_orders"), gold("fct_order_items")
mono = (fi.groupBy("order_id")
          .agg(F.countDistinct("seller_id").alias("n"), F.first("seller_id").alias("seller_id"))
          .filter("n = 1"))
passe = (fo.join(mono, "order_id")
           .filter((F.col("seller_id") == ligne.vendeur_principal)
                   & F.col("en_retard").isNotNull()
                   & ~F.col("livree_avant_transporteur") & ~F.col("statut_incoherent")
                   & (F.col("order_delivered_customer_date") < F.lit(ligne.order_approved_at)))
           .agg(F.count("*").alias("nb"), F.avg("en_retard").alias("taux"))
           .first())

print(f"commande {ligne.order_id}, approuvée le {ligne.order_approved_at}")
print(f"feature table : {ligne.vendeur_nb} livraisons passées, taux {ligne.vendeur_taux_retard:.4f}")
print(f"recalcul      : {passe.nb} livraisons passées, taux {passe.taux:.4f}")
assert ligne.vendeur_nb == passe.nb and abs(ligne.vendeur_taux_retard - passe.taux) < 1e-9, "Historique incorrect"

In [0]:
display(feat.filter("dans_population").limit(10))
display(
    feat.filter("dans_population")
    .select([F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 1).alias(c)
             for c in ["distance_max_km", "vendeur_taux_retard", "route_taux_retard",
                       "global_30j_taux_retard", "categorie_principale", "volume_total_cm3"]])
)